# Churn Prediction — Baseplate ML Walkthrough

This notebook walks through training a churn prediction model using Baseplate feature extractors.

> **Note:** Uses synthetic data from migration 0012 until real data exists in Phase 5+

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from src.features import build_feature_matrix

## 2. Load Data

In Phase 5+, load from Supabase. For now, generate synthetic data.

In [ ]:
# Synthetic data placeholder
# In Phase 5: query Supabase for real appointment/payment/engagement data
np.random.seed(42)
n_samples = 200
X = np.random.rand(n_samples, 15)  # 15 features from 3 extractors
y = (X[:, 4] < 0.3).astype(int)  # Low completion rate → churn
print(f'Samples: {n_samples}, Churn rate: {y.mean():.1%}')

## 3. Train + Cross-Validate

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(n_estimators=100, random_state=42)
cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='f1')
print(f'CV F1: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}')

model.fit(X_train, y_train)

## 4. Evaluate

In [ ]:
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

## 5. Feature Importance

In [ ]:
feature_names = [
    'appt_total', 'appt_completed', 'appt_cancelled', 'appt_no_shows',
    'appt_completion_rate', 'appt_cancel_rate', 'appt_no_show_rate',
    'pay_total', 'pay_completed', 'pay_failed', 'pay_revenue', 'pay_avg',
    'eng_login_count', 'eng_days_since_login', 'eng_total_events'
]
importance = pd.Series(model.feature_importances_, index=feature_names)
importance.sort_values(ascending=False).plot(kind='barh', figsize=(10, 6))

## 6. Save Model

In [ ]:
import pickle
from pathlib import Path
Path('../models').mkdir(exist_ok=True)
with open('../models/churn_model.pkl', 'wb') as f:
    pickle.dump(model, f)
print('Model saved to models/churn_model.pkl')